# 74 - Priority-trained Q50 evaluation on held-out PRO160

This single worker evaluates the completed **failure-prioritized** and **4-chunk-U20-prioritized** Q50 critics on all 160 frozen position-perturbation LIBERO-PRO identities used by notebook 68. It runs 320 new planner rollouts.

Both arms use the same inference contract as the previous Q50 evaluation: 64 candidates generated with 3 Euler steps, retain the top 16 by Q, Q-softmax-average their full 50-action chunks, execute the first 10 actions, and replan. The ordinary stock VLA is not rerun; each periodic table reuses its exact stored outcome on the same completed identities.

Videos, observation frames, and generated chunks are off. Compact executed trajectories and Q-planning diagnostics remain enabled. Results print every 10 identities for which both new arms have completed. Rerunning safely skips exact completed rows.

## 1. Setup

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Resolve the two step-6000 checkpoints

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

EPISODE_LIMIT = None  # set to 1 only for a two-rollout smoke test
CANDIDATE_BATCH_SIZE = 8  # lower only after a GPU-memory error
PRIORITY_ROOT = Path('/content/drive/MyDrive/pnp_qplanning_priority')
FAILURE_CHECKPOINT_PATH = None
U20_4CHUNK_CHECKPOINT_PATH = None

def resolve_priority_checkpoint(strategy, explicit):
    if explicit is not None:
        path = Path(explicit)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    matches = sorted(PRIORITY_ROOT.glob(
        f'pcpcds-*/q50_priority_{strategy}_full/checkpoint_step_006000.pt'))
    if len(matches) != 1:
        raise ValueError(
            f'Expected exactly one step-6000 {strategy} checkpoint; found ' +
            f'{len(matches)}: {[str(path) for path in matches]}. Set its path explicitly.')
    return matches[0]

FAILURE_CHECKPOINT_PATH = resolve_priority_checkpoint(
    'failure', FAILURE_CHECKPOINT_PATH)
U20_4CHUNK_CHECKPOINT_PATH = resolve_priority_checkpoint(
    'u20_4chunk', U20_4CHUNK_CHECKPOINT_PATH)
if FAILURE_CHECKPOINT_PATH.parents[1] != U20_4CHUNK_CHECKPOINT_PATH.parents[1]:
    raise ValueError('The two critics must come from the same pcpcds-* snapshot directory')

print({
    'worker': 'single full PRO160 worker',
    'identities': 160 if EPISODE_LIMIT is None else EPISODE_LIMIT,
    'new_rollouts': 320 if EPISODE_LIMIT is None else 2 * EPISODE_LIMIT,
    'failure_checkpoint': str(FAILURE_CHECKPOINT_PATH),
    'u20_4chunk_checkpoint': str(U20_4CHUNK_CHECKPOINT_PATH),
    'candidate_batch_size': CANDIDATE_BATCH_SIZE,
    'periodic_print_every_complete_identities': 10,
    'historical_stock': 'reuse exact matched notebook-68 rows; do not rerun',
})

## 3. Run both priority critics

In [ ]:
from pnp.qplanning_priority_eval_experiment import run_qplanning_priority_heldout160

report = run_qplanning_priority_heldout160(
    failure_checkpoint_path=FAILURE_CHECKPOINT_PATH,
    u20_4chunk_checkpoint_path=U20_4CHUNK_CHECKPOINT_PATH,
    episode_limit=EPISODE_LIMIT,
    candidate_batch_size=CANDIDATE_BATCH_SIZE,
)
report

## 4. Audit persisted planner settings

In [ ]:
from pnp.qplanning_priority_eval_experiment import validate_qplanning_priority_sentinel

validate_qplanning_priority_sentinel(
    failure_checkpoint_id=report['checkpoint_ids']['failure'],
    u20_4chunk_checkpoint_id=report['checkpoint_ids']['u20_4chunk'],
    experiment=report['experiment'],
)